# Random Forests

Random Forests are powerful ensemble learning method used for both classification and regression. they overcome the problem of major
Decision Trees-its tendency to overfit, by training not one but many Decision Trees and combining their results.
This "Forest" of trees is more robust and accurate than a single tree.


### The Core Idea: The Wisdom of Crowds.
Think of a Random Forest as a group of experts, where each expert is a single decision tree. When a new data point needs to be classified, each expert (tree in the forest) makes theur own prediction. For classification problem, the final prediction is the one that gets the most "votes" from the trees.
For a regression problem, the final prediction is the average of all the trees' predictions.

The key to why this works os that the trees are intentionally made to be different from one another. this 'randomness' is introduced in teo main ways:

1. **Bagging (Bootstrap Agregating)**: Each tree in the forest is trainined on a different random sample of the original training data. This sampling is done with replacement. Meaning a single data point can appear multiple times in  a single sample, and some points may not appear at all - this process is called Bootstrapping.

2. **Feature Randomness** when a tree is built, at each split, it doesn't consider all available features. Instead, it only considers a random subset of 
features. This forces the trees to be diverse and prevents them from becoming too similar (all of them focusing on the most dominant feature.)

This combination of Bootstrapping and feature randomness ensures that each tree in the forest is unique. By averaging their predictions, the Random Forest model *reduces the variance* of individual trees, leading to a much more stable and accurate prediction.

In [1]:
import numpy as np
from collections import Counter
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# The Node class from our previous Decision Tree implementation
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None

# The DecisionTree class from our previous implementation
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=100, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        best_feature_idx, best_threshold = self._best_split(X, y)

        left_idxs = np.where(X[:, best_feature_idx] <= best_threshold)[0]
        right_idxs = np.where(X[:, best_feature_idx] > best_threshold)[0]

        left_child = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right_child = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)

        return Node(best_feature_idx, best_threshold, left_child, right_child)

    def _best_split(self, X, y):
        best_gini = float('inf')
        split_idx, split_threshold = None, None

        # Randomly select a subset of features to consider
        feature_indices = np.random.choice(X.shape[1], self.n_features, replace=False)

        for feature_idx in feature_indices:
            X_column = X[:, feature_idx]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                left_y = y[X_column <= threshold]
                right_y = y[X_column > threshold]
                
                if len(left_y) == 0 or len(right_y) == 0:
                    continue

                n_left, n_right = len(left_y), len(right_y)
                n_total = len(y)

                gini_left = self._gini_impurity(left_y)
                gini_right = self._gini_impurity(right_y)

                weighted_gini = (n_left / n_total) * gini_left + (n_right / n_total) * gini_right

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    split_idx = feature_idx
                    split_threshold = threshold

        return split_idx, split_threshold

    def _gini_impurity(self, y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities**2)

    def _most_common_label(self, y):
        counter = Counter(y)
        most_common = counter.most_common(1)[0][0]
        return most_common

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

# --- The New RandomForestClassifier Class ---
class RandomForestClassifier:
    def __init__(self, n_trees=10, min_samples_split=2, max_depth=100, n_features=None):
        self.n_trees = n_trees
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.trees = [] # A list to hold all the DecisionTree objects

    def fit(self, X, y):
        # Clear any existing trees
        self.trees = []
        # Get the number of features if not specified
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        
        # Loop to create and train each tree
        for _ in range(self.n_trees):
            # Create a new Decision Tree with our parameters
            tree = DecisionTree(
                min_samples_split=self.min_samples_split,
                max_depth=self.max_depth,
                n_features=self.n_features
            )
            # Create a random subset of the training data (bootstrap sample)
            X_sample, y_sample = self._bootstrap_sample(X, y)
            # Train the tree on this random sample
            tree.fit(X_sample, y_sample)
            # Add the trained tree to our forest
            self.trees.append(tree)

    def _bootstrap_sample(self, X, y):
        """
        Creates a random sample of the data with replacement.
        """
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def predict(self, X):
        """
        Makes a final prediction by taking a majority vote from all trees.
        """
        # Get the predictions from each individual tree
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # Transpose to get predictions per data point
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        # For each data point, find the most common prediction (the majority vote)
        y_pred = np.array([self._most_common_label(pred) for pred in tree_preds])
        return y_pred

    def _most_common_label(self, y):
        counter = Counter(y)
        most_common = counter.most_common(1)[0][0]
        return most_common


# --- Putting it all together: A simple classification example ---
# 1. Generate synthetic classification data
X, y = make_blobs(n_samples=500, centers=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Instantiate and train the Random Forest model
# We'll use 10 trees
rf_model = RandomForestClassifier(n_trees=10, max_depth=5, n_features=2)
rf_model.fit(X_train, y_train)

# 3. Make predictions on the test set
y_pred = rf_model.predict(X_test)

# 4. Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Model Accuracy: {accuracy * 100:.2f}%")


Random Forest Model Accuracy: 100.00%
